<a href="https://colab.research.google.com/github/rakshamishra54/sql_analytics_notebooks/blob/main/Netflix_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import kagglehub, sqlite3, pandas as pd

path = kagglehub.dataset_download("shivamb/netflix-shows")
nf = pd.read_csv(f'{path}/netflix_titles.csv')
nf['date_added'] = pd.to_datetime(nf['date_added'].str.strip(), errors='coerce').dt.strftime('%Y-%m-%d')

Using Colab cache for faster access to the 'netflix-shows' dataset.


In [26]:
netflix = sqlite3.connect('/content/netflix.db')
nf.to_sql('netflix', netflix, if_exists='replace', index=False)

def run(query, conn):
    return pd.read_sql(query, conn)

run("SELECT COUNT(*) AS rows_loaded FROM netflix", netflix)

,rows_loaded
0,8807


In [27]:
run("SELECT type, COUNT(*) AS total FROM netflix GROUP BY type", netflix)

,type,total
0,Movie,6131
1,TV Show,2676


In [28]:
## Count the number of movies vs TV shows
run("""
SELECT type ,
COUNT(*) AS total
FROM netflix
GROUP BY type
""", netflix)

,type,total
0,Movie,6131
1,TV Show,2676


In [29]:
## Find the most common rating for movies and TV shows
run("""
SELECT type, rating, total_count
FROM (SELECT type,rating, COUNT(*) AS total_count,
RANK() OVER (PARTITION BY type ORDER BY COUNT(*) DESC) AS ranking
FROM netflix
GROUP BY type, rating
) AS t
WHERE ranking = 1
""", netflix)

,type,rating,total_count
0,Movie,TV-MA,2062
1,TV Show,TV-MA,1145


In [30]:
## List all movies released in a specific year
run("""
SELECT * FROM netflix
WHERE type = 'Movie'AND release_year = 2020

""", netflix)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,None,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s17,Movie,Europe's Most Dangerous Man: Otto Skorzeny in ...,"Pedro de Echave García, Pablo Azorín Williams",None,None,2021-09-22,2020,TV-MA,67 min,"Documentaries, International Movies",Declassified documents reveal the post-WWII li...
2,s79,Movie,Tughlaq Durbar,Delhiprasad Deenadayalan,"Vijay Sethupathi, Parthiban, Raashi Khanna",None,2021-09-11,2020,TV-14,145 min,"Comedies, Dramas, International Movies",A budding politician has devious plans to rise...
3,s85,Movie,Omo Ghetto: the Saga,"JJC Skillz, Funke Akindele","Funke Akindele, Ayo Makun, Chioma Chukwuka Akp...",Nigeria,2021-09-10,2020,TV-MA,147 min,"Action & Adventure, Comedies, Dramas",Twins are reunited as a good-hearted female ga...
4,s104,Movie,Shadow Parties,Yemi Amodu,"Jide Kosoko, Omotola Jalade-Ekeinde, Yemi Blaq...",None,2021-09-06,2020,TV-MA,117 min,"Dramas, International Movies, Thrillers",A family faces destruction in a long-running c...
...,...,...,...,...,...,...,...,...,...,...,...,...
512,s3047,Movie,All the Freckles in the World,Yibrán Asuad,"Hánssel Casillas, Loreto Peralta, Andrea Sutto...",Mexico,2020-01-03,2020,TV-14,90 min,"Comedies, International Movies, Romantic Movies",Thirteen-year-old José Miguel is immune to 199...
513,s3061,Movie,Ghost Stories,"Anurag Kashyap, Dibakar Banerjee, Karan Johar,...","Janhvi Kapoor, Sobhita Dhulipala, Sukant Goel,...",India,2020-01-01,2020,TV-MA,145 min,"Horror Movies, International Movies, Thrillers","The directors of Emmy-nominated ""Lust Stories""..."
514,s5973,Movie,#cats_the_mewvie,Michael Margolis,None,Canada,2020-02-05,2020,TV-14,90 min,"Documentaries, International Movies",This pawesome documentary explores how our fel...
515,s7595,Movie,Norm of the North: Family Vacation,Anthony Bell,"Andrew Toth, Lisa Durupt, Jonathan Holmes, Pau...",United States,2020-05-25,2020,TV-Y7,89 min,Children & Family Movies,"Stressed by his duties as king and father, Nor..."


In [31]:
## Find the top 5 countries with the most content on Netflix
run("""
SELECT country,
COUNT(*) AS totalContent
FROM netflix
GROUP BY country
ORDER BY totalContent DESC
LIMIT 5
""" , netflix)

,country,totalContent
0,United States,2818
1,India,972
2,None,831
3,United Kingdom,419
4,Japan,245


In [32]:
##Identify the longest movie
run("""
SELECT title, duration
FROM netflix
WHERE type = 'Movie' AND duration IS NOT NULL
ORDER BY CAST(SUBSTR(duration, 1, INSTR(duration, ' ') - 1) AS INTEGER) DESC
LIMIT 5
""", netflix)

,title,duration
0,Black Mirror: Bandersnatch,312 min
1,Headspace: Unwind Your Mind,273 min
2,The School of Mischief,253 min
3,No Longer kids,237 min
4,Lock Your Girls In,233 min


In [33]:
run("""
SELECT title, duration,
CAST(duration AS INTEGER) AS duration_minutes
FROM netflix
WHERE type = 'Movie' AND duration IS NOT NULL
ORDER BY duration_minutes DESC
LIMIT 10;
""", netflix)

,title,duration,duration_minutes
0,Black Mirror: Bandersnatch,312 min,312
1,Headspace: Unwind Your Mind,273 min,273
2,The School of Mischief,253 min,253
3,No Longer kids,237 min,237
4,Lock Your Girls In,233 min,233
5,Raya and Sakina,230 min,230
6,Once Upon a Time in America,229 min,229
7,Sangam,228 min,228
8,Lagaan,224 min,224
9,Jodhaa Akbar,214 min,214


In [34]:
## Find the content added in the last 5 years

In [35]:
##Find all the movies/TV shows by director 'Rajiv Chilaka'!
run("""
SELECT director, title
FROM netflix
WHERE director LIKE '%Rajiv Chilaka%'
""", netflix)

,director,title
0,Rajiv Chilaka,Chhota Bheem - Neeli Pahaadi
1,Rajiv Chilaka,Chhota Bheem & Ganesh
2,Rajiv Chilaka,Chhota Bheem & Krishna: Mayanagari
3,Rajiv Chilaka,Chhota Bheem & Krishna: Pataliputra- City of t...
4,Rajiv Chilaka,Chhota Bheem And The Broken Amulet
5,Rajiv Chilaka,Chhota Bheem And The Crown of Valhalla
6,Rajiv Chilaka,Chhota Bheem and the Incan Adventure
7,Rajiv Chilaka,Chhota Bheem and The ShiNobi Secret
8,Rajiv Chilaka,Chhota Bheem Aur Hanuman
9,Rajiv Chilaka,Chhota Bheem aur Krishna


In [36]:
## List all TV shows with more than 5 seasons
run("""
SELECT *
FROM netflix
WHERE type = 'TV Show'
AND
CAST(SUBSTR(duration , 1, INSTR(duration , ' ') - 1) AS INTEGER) > 5
""", netflix)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,2021-09-24,2021,TV-14,9 Seasons,"British TV Shows, Reality TV",A talented batch of amateur bakers face off in...
1,s56,TV Show,Nailed It,None,"Nicole Byer, Jacques Torres",United States,2021-09-15,2021,TV-PG,6 Seasons,Reality TV,Home bakers with a terrible track record take ...
2,s66,TV Show,Numberblocks,None,"Beth Chalmers, David Holt, Marcel McCalla, Ter...",United Kingdom,2021-09-15,2021,TV-Y,6 Seasons,Kids' TV,"In a place called Numberland, math adds up to ..."
3,s68,TV Show,Saved by the Bell,None,"Mark-Paul Gosselaar, Tiffani Thiessen, Mario L...",United States,2021-09-15,1994,TV-PG,9 Seasons,"Kids' TV, TV Comedies","From middle school to college, best friends Za..."
4,s83,TV Show,Lucifer,None,"Tom Ellis, Lauren German, Kevin Alejandro, D.B...",United States,2021-09-10,2021,TV-14,6 Seasons,"Crime TV Shows, TV Comedies, TV Dramas","Bored with being the Lord of Hell, the devil r..."
...,...,...,...,...,...,...,...,...,...,...,...,...
94,s8190,TV Show,The Andy Griffith Show,None,"Andy Griffith, Ron Howard, Don Knotts, Frances...",United States,2017-07-01,1967,TV-G,8 Seasons,"Classic & Cult TV, TV Comedies",Homespun humor and easygoing Sheriff Andy Tayl...
95,s8379,TV Show,The L Word,None,"Jennifer Beals, Leisha Hailey, Laurel Holloman...","Canada, United States",2018-03-20,2009,TV-MA,6 Seasons,"Romantic TV Shows, TV Dramas",This trailblazing series follows the tangled l...
96,s8443,TV Show,The Office (U.S.),None,"Steve Carell, John Krasinski, Jenna Fischer, R...",United States,2016-01-01,2012,TV-14,9 Seasons,TV Comedies,This hit comedy chronicles the foibles of disg...
97,s8558,TV Show,The West Wing,None,"Martin Sheen, Rob Lowe, Allison Janney, John S...",United States,2015-12-25,2005,TV-14,7 Seasons,TV Dramas,This powerful political epic chronicles the tr...


In [37]:
## List all TV shows with more than 5 seasons (using diff method as above )
run("""
SELECT title,duration,
CAST(duration AS INTEGER) AS seasons
FROM netflix
WHERE type = 'TV Show'AND duration IS NOT NULL AND CAST(duration AS INTEGER) > 5
ORDER BY seasons DESC;
""", netflix)

,title,duration,seasons
0,Grey's Anatomy,17 Seasons,17
1,Supernatural,15 Seasons,15
2,NCIS,15 Seasons,15
3,Heartland,13 Seasons,13
4,COMEDIANS of the world,13 Seasons,13
...,...,...,...
94,West Coast Customs,6 Seasons,6
95,Father Brown,6 Seasons,6
96,Gossip Girl,6 Seasons,6
97,Republic of Doyle,6 Seasons,6


In [38]:
#Count the number of content items in each genre
run("""
SELECT listed_in ,COUNT(*) AS totalcontent
FROM netflix
GROUP BY listed_in
ORDER BY totalcontent DESC
""" , netflix)

,listed_in,totalcontent
0,"Dramas, International Movies",362
1,Documentaries,359
2,Stand-Up Comedy,334
3,"Comedies, Dramas, International Movies",274
4,"Dramas, Independent Movies, International Movies",252
...,...,...
509,"Action & Adventure, Classic Movies, Internatio...",1
510,"Action & Adventure, Children & Family Movies, ...",1
511,"Action & Adventure, Children & Family Movies, ...",1
512,"Action & Adventure, Anime Features, Horror Movies",1


In [39]:
##Find each year and the average numbers of content release by India on netflix.  return top 5 year with highest avg content release

In [40]:
## List all movies that are crime thrillers
run("""
SELECT * FROM netflix
WHERE listed_in LIKE '%Crime%' OR '%Thrillers%'
""", netflix)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",None,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
1,s11,TV Show,"Vendetta: Truth, Lies and The Mafia",None,None,None,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, Docuseries, International TV S...","Sicily boasts a bold ""Anti-Mafia"" coalition. B..."
2,s12,TV Show,Bangkok Breaking,Kongkiat Komesiri,"Sukollawat Kanarot, Sushar Manaying, Pavarit M...",None,2021-09-23,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...","Struggling to earn a living in Bangkok, a man ..."
3,s15,TV Show,Crime Stories: India Detectives,None,None,None,2021-09-22,2021,TV-MA,1 Season,"British TV Shows, Crime TV Shows, Docuseries",Cameras following Bengaluru police on the job ...
4,s18,TV Show,Falsa identidad,None,"Luis Ernesto Franco, Camila Sodi, Sergio Goyri...",Mexico,2021-09-22,2020,TV-MA,2 Seasons,"Crime TV Shows, Spanish-Language TV Shows, TV ...",Strangers Diego and Isabel flee their home in ...
...,...,...,...,...,...,...,...,...,...,...,...,...
465,s8534,TV Show,The Traffickers,None,Nelufar Hedayat,United States,2018-01-22,2016,TV-MA,1 Season,"Crime TV Shows, Docuseries","In this hard-hitting series, journalist Nelufa..."
466,s8589,TV Show,Tientsin Mystic,None,"Li Xian, Zhang Ming En, Wang Zixuan, Chen Yumi...",China,2018-02-09,2017,TV-14,1 Season,"Crime TV Shows, International TV Shows, TV Dramas","In an otherworldly river town, a policeman fol..."
467,s8594,TV Show,Timeline,None,"Santiwithi Phrombut, Jamorn Kijsawapak, Orrawa...",None,2018-02-23,2014,TV-14,1 Season,"Crime TV Shows, Docuseries, International TV S...","Firsthand accounts, re-enactments and computer..."
468,s8645,TV Show,Twin Peaks,None,"Kyle MacLachlan, Michael Ontkean, Mädchen Amic...",United States,2017-07-01,1990,TV-14,2 Seasons,"Classic & Cult TV, Crime TV Shows, TV Dramas","""Who killed Laura Palmer?"" is the question on ..."


In [41]:
## Find all content without a director
run("""
SELECT * FROM netflix
WHERE director IS NULL
""", netflix)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s2,TV Show,Blood & Water,None,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
1,s4,TV Show,Jailbirds New Orleans,None,None,None,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
2,s5,TV Show,Kota Factory,None,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
3,s11,TV Show,"Vendetta: Truth, Lies and The Mafia",None,None,None,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, Docuseries, International TV S...","Sicily boasts a bold ""Anti-Mafia"" coalition. B..."
4,s15,TV Show,Crime Stories: India Detectives,None,None,None,2021-09-22,2021,TV-MA,1 Season,"British TV Shows, Crime TV Shows, Docuseries",Cameras following Bengaluru police on the job ...
...,...,...,...,...,...,...,...,...,...,...,...,...
2629,s8796,TV Show,Yu-Gi-Oh! Arc-V,None,"Mike Liscio, Emily Bauer, Billy Bob Thompson, ...","Japan, Canada",2018-05-01,2015,TV-Y7,2 Seasons,"Anime Series, Kids' TV",Now that he's discovered the Pendulum Summonin...
2630,s8797,TV Show,Yunus Emre,None,"Gökhan Atalay, Payidar Tüfekçioglu, Baran Akbu...",Turkey,2017-01-17,2016,TV-PG,2 Seasons,"International TV Shows, TV Dramas","During the Mongol invasions, Yunus Emre leaves..."
2631,s8798,TV Show,Zak Storm,None,"Michael Johnston, Jessica Gee-George, Christin...","United States, France, South Korea, Indonesia",2018-09-13,2016,TV-Y7,3 Seasons,Kids' TV,Teen surfer Zak Storm is mysteriously transpor...
2632,s8801,TV Show,Zindagi Gulzar Hai,None,"Sanam Saeed, Fawad Khan, Ayesha Omer, Mehreen ...",Pakistan,2016-12-15,2012,TV-PG,1 Season,"International TV Shows, Romantic TV Shows, TV ...","Strong-willed, middle-class Kashaf and carefre..."


In [42]:
##Find how many movies actor 'Salman Khan' appeared in last 10 years
run("""
SELECT COUNT(*) AS salman_movies
FROM netflix
WHERE type = 'Movie'
  AND "cast" LIKE '%Salman Khan%'
  AND release_year > (SELECT MAX(release_year) FROM netflix) - 10
""", netflix)

,salman_movies
0,3


In [43]:
##Categorize the content based on the presence of the keywords 'kill' and 'violence' in the description field. Label content containing these keywords as 'Bad' and all other content as 'Good'. Count how many items fall into each category
run("""
SELECT category, COUNT(*) AS content_count
FROM (
    SELECT *,
        CASE
            WHEN description LIKE '%kill%' OR description LIKE '%violence%' THEN 'Bad'
            ELSE 'Good'
        END AS category
    FROM netflix
)
GROUP BY category
""", netflix)

,category,content_count
0,Bad,342
1,Good,8465
